# NB01 — Corpus Discovery & Inventory

This notebook explores the KDK repository to build a complete picture of what data is available for a RAG pipeline.

**Exploration thread:**
1. Full file scan and extension inventory
2. Exclusion strategy (noise removal)
3. Distribution by zone (`core`, `map`, `docs`, `test`, ...)
4. Content family classification
5. Documentation deep-dive: structure, headings, special content, token budget
6. JSON deep-dive: functional types, RAG value, token cost
7. Source code signals: semantic density in JS/Vue files
8. Summary and decisions carried forward

In [ ]:
import os
import re
import json
from pathlib import Path
from collections import Counter, defaultdict

BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / "data" / "kdk").exists():
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data" / "kdk"
DOCS_DIR = DATA_DIR / "docs"
print(f"Data directory: {DATA_DIR.resolve()}")

In [ ]:
# ── Shared helpers ──
EXCLUDED_DIRS = {
    "node_modules", ".git", "dist", "build", ".output",
    "coverage", ".nyc_output", ".c8", "__pycache__", ".cache", ".github",
}
EXCLUDED_FILES = {"yarn.lock", "CHANGELOG.md"}
EXCLUDED_EXTENSIONS = {".png", ".gif", ".tif", ".svg", ".lock"}
RATIO_TOKENS = 4  # ~1 token per 4 chars for English technical text

def scan_files(root, excluded_dirs):
    files = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in excluded_dirs]
        for f in filenames:
            files.append(Path(dirpath) / f)
    return files

def fmt_size(nbytes):
    for unit in ["B", "KB", "MB", "GB"]:
        if nbytes < 1024:
            return f"{nbytes:.1f} {unit}"
        nbytes /= 1024
    return f"{nbytes:.1f} TB"

def read_text(path):
    return path.read_text(encoding="utf-8", errors="ignore")

def estimate_tokens(text):
    return len(text) // RATIO_TOKENS

---
## 1. Full file scan

In [ ]:
all_files_raw = scan_files(DATA_DIR, EXCLUDED_DIRS)
print(f"Files found (after directory exclusions): {len(all_files_raw)}")

inventory = {}
for f in all_files_raw:
    ext = f.suffix.lower() if f.suffix else "(no ext)"
    if ext not in inventory:
        inventory[ext] = {"count": 0, "size": 0}
    inventory[ext]["count"] += 1
    try:
        inventory[ext]["size"] += f.stat().st_size
    except OSError:
        pass

print(f"\n{'Extension':<20} {'Count':>8} {'Size':>12}")
print("-" * 42)
for ext, data in sorted(inventory.items(), key=lambda x: x[1]["size"], reverse=True):
    print(f"{ext:<20} {data['count']:>8} {fmt_size(data['size']):>12}")

total_raw_size = sum(d["size"] for d in inventory.values())
print(f"\nTotal: {len(all_files_raw)} files, {fmt_size(total_raw_size)}")
print(f"Distinct extensions: {len(inventory)}")

---
## 2. Exclusion strategy

Remove files with no semantic value for RAG: lock files, binary images, auto-generated changelogs.

In [ ]:
all_files = [
    f for f in all_files_raw
    if f.name not in EXCLUDED_FILES
    and f.suffix.lower() not in EXCLUDED_EXTENSIONS
]

excluded_count = len(all_files_raw) - len(all_files)
retained_size = sum(f.stat().st_size for f in all_files)
excluded_size = total_raw_size - retained_size

print(f"Before exclusions: {len(all_files_raw)}")
print(f"Excluded:          {excluded_count} ({fmt_size(excluded_size)})")
print(f"Retained:          {len(all_files)} ({fmt_size(retained_size)})")

# Retained breakdown
ext_counter = Counter()
ext_size = Counter()
for f in all_files:
    ext = f.suffix.lower() if f.suffix else "(no ext)"
    ext_counter[ext] += 1
    ext_size[ext] += f.stat().st_size

print(f"\n{'Extension':<20} {'Count':>8} {'Size':>12}")
print("-" * 42)
for ext, size in ext_size.most_common():
    print(f"{ext:<20} {ext_counter[ext]:>8} {fmt_size(size):>12}")

---
## 3. Distribution by zone

In [ ]:
by_zone = defaultdict(list)
for f in all_files:
    rel = f.relative_to(DATA_DIR)
    zone = rel.parts[0] if len(rel.parts) > 1 else "(root)"
    by_zone[zone].append(f)

print(f"{'Zone':<12} {'Files':>8} {'Size':>12} {'Top extensions'}")
print("-" * 80)
for zone, files in sorted(by_zone.items(), key=lambda x: len(x[1]), reverse=True):
    size = sum(f.stat().st_size for f in files)
    exts = Counter(f.suffix.lower() if f.suffix else "(no ext)" for f in files)
    top = ", ".join(f"{e}:{n}" for e, n in exts.most_common(4))
    print(f"{zone:<12} {len(files):>8} {fmt_size(size):>12} {top}")

---
## 4. Content family classification

Group files into processing families that will each need a different chunking approach.

In [ ]:
def classify_file(path):
    rel = path.relative_to(DATA_DIR)
    parts = rel.parts
    ext = path.suffix.lower()
    if parts[0] == "docs" and ext == ".md":
        return "documentation_markdown"
    if ext == ".vue":
        return "vue_sfc"
    if path.name == "index.js":
        return "barrel_index_js"
    if parts[0] == "test":
        return "tests"
    if ext in {".js", ".mjs", ".cjs"}:
        return "code_js"
    if ext == ".json":
        return "json"
    if ext in {".md", ".txt"}:
        return "text_outside_docs"
    if ext in {".html", ".ejs"}:
        return "templates"
    if ext in {".xml", ".drawio"}:
        return "diagrams_and_specs"
    return "other"

by_family = defaultdict(list)
for f in all_files:
    by_family[classify_file(f)].append(f)

print("Content families:")
for family, files in sorted(by_family.items(), key=lambda x: len(x[1]), reverse=True):
    size = sum(f.stat().st_size for f in files)
    print(f"  {family:<24} {len(files):>4} files  {fmt_size(size):>10}")

---
## 5. Documentation deep-dive (`docs/`)

The `docs/` directory is the most structured part of the corpus. We analyze its heading hierarchy,
section sizes, special content, and token budget to inform the Markdown chunking strategy.

In [ ]:
# ── 5a. File tree and family profiles ──
md_files = sorted(DOCS_DIR.rglob("*.md"))
print(f"Markdown files in docs/: {len(md_files)}")
print(f"Total size: {sum(f.stat().st_size for f in md_files) / 1024:.1f} KB")

def doc_family(path):
    rel = path.relative_to(DOCS_DIR)
    return rel.parts[0] if len(rel.parts) > 1 else "(root)"

family_stats = defaultdict(lambda: {"files": 0, "lines": 0, "chars": 0, "h2": 0, "h3": 0, "code_blocks": 0})
for f in md_files:
    content = f.read_text(encoding="utf-8")
    fam = doc_family(f)
    family_stats[fam]["files"] += 1
    family_stats[fam]["lines"] += len(content.splitlines())
    family_stats[fam]["chars"] += len(content)
    family_stats[fam]["h2"] += len(re.findall(r"^##\s+", content, re.MULTILINE))
    family_stats[fam]["h3"] += len(re.findall(r"^###\s+", content, re.MULTILINE))
    family_stats[fam]["code_blocks"] += len(re.findall(r"^```", content, re.MULTILINE)) // 2

print(f"\n{'Family':<14} {'Files':>6} {'Lines':>7} {'H2':>5} {'H3':>5} {'Code':>5} {'~Tokens':>9}")
print("-" * 60)
for fam, row in sorted(family_stats.items()):
    tokens = row["chars"] // RATIO_TOKENS
    print(f"{fam:<14} {row['files']:>6} {row['lines']:>7} {row['h2']:>5} {row['h3']:>5} {row['code_blocks']:>5} {tokens:>9}")

In [ ]:
# ── 5b. Heading hierarchy ──
def extract_sections(filepath):
    content = filepath.read_text(encoding="utf-8")
    lines = content.splitlines()
    sections = []
    current = None
    start_line = 0
    in_code_block = False
    for i, line in enumerate(lines):
        if line.strip().startswith("```"):
            in_code_block = not in_code_block
            continue
        if in_code_block:
            continue
        match = re.match(r"^(#{1,6})\s+(.+)", line)
        if match:
            if current is not None:
                current["lines"] = i - start_line
                sections.append(current)
            level = len(match.group(1))
            current = {"level": level, "title": match.group(2).strip(), "line_start": i + 1, "lines": 0}
            start_line = i
    if current is not None:
        current["lines"] = len(lines) - start_line
        sections.append(current)
    return sections

all_sections = {}
level_counts = Counter()
for f in md_files:
    secs = extract_sections(f)
    all_sections[f] = secs
    for s in secs:
        level_counts[s["level"]] += 1

print("Heading distribution:")
for lvl in sorted(level_counts):
    bar = "#" * lvl
    print(f"  {bar:<6} (H{lvl}): {level_counts[lvl]:>4} sections")
total_sections = sum(level_counts.values())
print(f"\nTotal: {total_sections} sections across {len(md_files)} files")

In [ ]:
# ── 5c. Section size distribution and chunking simulation (## level) ──
def simulate_h2_chunks(filepath, sections):
    content = filepath.read_text(encoding="utf-8")
    lines = content.splitlines()
    if not sections:
        return [{"file": str(filepath.relative_to(DOCS_DIR)), "title": filepath.stem,
                 "tokens": len(content) // RATIO_TOKENS}]
    chunks = []
    doc_title = next((s["title"] for s in sections if s["level"] == 1), filepath.stem)
    current = None
    start = 0
    for s in sections:
        if s["level"] <= 2:
            if current is not None:
                end = s["line_start"] - 1
                text = chr(10).join(lines[start:end])
                current["tokens"] = len(text) // RATIO_TOKENS
                chunks.append(current)
            current = {"file": str(filepath.relative_to(DOCS_DIR)),
                       "doc_title": doc_title, "title": s["title"]}
            start = s["line_start"] - 1
    if current is not None:
        text = chr(10).join(lines[start:])
        current["tokens"] = len(text) // RATIO_TOKENS
        chunks.append(current)
    return chunks

all_chunks = []
for f in md_files:
    all_chunks.extend(simulate_h2_chunks(f, all_sections.get(f, [])))

chunk_tokens = sorted(c["tokens"] for c in all_chunks)
small = [c for c in all_chunks if c["tokens"] < 50]
large = [c for c in all_chunks if c["tokens"] > 1500]

print(f"H2-level chunking simulation: {len(all_chunks)} chunks")
print(f"  Min:    {min(chunk_tokens):>6}")
print(f"  Median: {chunk_tokens[len(chunk_tokens)//2]:>6}")
print(f"  Mean:   {sum(chunk_tokens)//len(chunk_tokens):>6}")
print(f"  Max:    {max(chunk_tokens):>6}")
print(f"  Too small (<50 tokens): {len(small)} — candidates for merging")
print(f"  Too large (>1500 tokens): {len(large)} — candidates for H3 re-split")

if large:
    print("\n  Largest chunks:")
    for c in sorted(large, key=lambda x: x["tokens"], reverse=True):
        print(f"    {c['tokens']:>5} tok | {c['title']:<35} | {c['file']}")

In [ ]:
# ── 5d. Special content detection ──
patterns = {
    "code_blocks": re.compile(r"^```", re.MULTILINE),
    "mermaid": re.compile(r"^```mermaid", re.MULTILINE),
    "tables": re.compile(r"^\|.*\|.*\|", re.MULTILINE),
    "admonitions": re.compile(r"^:::\s*(tip|warning|danger|info|details)", re.MULTILINE),
    "internal_links": re.compile(r"\[([^\]]+)\]\(\./([^)]+)\)"),
    "images": re.compile(r"!\[([^\]]*)\]\(([^)]+)\)"),
}

special_results = defaultdict(lambda: {"files": set(), "total": 0})
for f in md_files:
    content = f.read_text(encoding="utf-8")
    for name, pattern in patterns.items():
        matches = pattern.findall(content)
        if matches:
            special_results[name]["files"].add(str(f.relative_to(DOCS_DIR)))
            special_results[name]["total"] += len(matches)

print(f"{'Content type':<25} {'Count':>8} {'Files':>8}")
print("-" * 44)
for name, data in sorted(special_results.items()):
    count = data["total"] // 2 if name == "code_blocks" else data["total"]
    print(f"{name:<25} {count:>8} {len(data['files']):>8}")

In [ ]:
# ── 5e. Token budget ──
total_doc_chars = sum(len(f.read_text(encoding="utf-8")) for f in md_files)
total_doc_tokens = total_doc_chars // RATIO_TOKENS
print(f"Total documentation tokens: ~{total_doc_tokens:,}")
print(f"At 500 tokens/chunk: ~{total_doc_tokens // 500} chunks")
print(f"At 1000 tokens/chunk: ~{total_doc_tokens // 1000} chunks")

---
## 6. JSON deep-dive

JSON files are 32% of the corpus by size but vary wildly in RAG value.
We classify them functionally and assess which to include.

In [ ]:
# ── 6a. JSON inventory and functional classification ──
json_files = sorted(f for f in all_files if f.suffix.lower() == ".json")
print(f"JSON files: {len(json_files)}, total size: {fmt_size(sum(f.stat().st_size for f in json_files))}")

def classify_json(path):
    rel = path.relative_to(DATA_DIR)
    parts = rel.parts
    rel_str = str(rel).lower()
    name = path.name.lower()
    if name == "package.json":
        return "package_tooling"
    if "schemas" in parts or name.endswith("create.json") or name.endswith("update.json"):
        return "schemas_validation"
    if "/i18n/" in rel_str or name.endswith("_fr.json") or name.endswith("_en.json"):
        return "i18n_translations"
    if parts[0] == "test":
        return "test_fixtures" if "config" not in parts else "test_config"
    if parts[0] == "docs":
        return "docs_meta"
    return "other_json"

json_by_type = defaultdict(list)
for f in json_files:
    json_by_type[classify_json(f)].append(f)

print(f"\n{'JSON type':<24} {'Files':>6} {'Size':>10} {'Examples'}")
print("-" * 90)
for cat, files in sorted(json_by_type.items(), key=lambda x: len(x[1]), reverse=True):
    size = sum(f.stat().st_size for f in files)
    examples = ", ".join(str(f.relative_to(DATA_DIR)) for f in files[:2])
    print(f"{cat:<24} {len(files):>6} {fmt_size(size):>10} {examples}")

In [ ]:
# ── 6b. RAG value prioritization ──
RAG_PRIORITY = {
    "schemas_validation": "high",
    "i18n_translations": "medium",
    "docs_meta": "medium",
    "test_config": "low",
    "test_fixtures": "low",
    "package_tooling": "exclude",
    "other_json": "evaluate",
}

by_priority = defaultdict(lambda: {"files": 0, "size": 0, "types": Counter()})
for cat, files in json_by_type.items():
    prio = RAG_PRIORITY.get(cat, "evaluate")
    by_priority[prio]["files"] += len(files)
    by_priority[prio]["size"] += sum(f.stat().st_size for f in files)
    by_priority[prio]["types"][cat] += len(files)

print(f"{'Priority':<16} {'Files':>6} {'Size':>10} {'Categories'}")
print("-" * 70)
for prio, row in sorted(by_priority.items(), key=lambda x: x[1]["size"], reverse=True):
    cats = ", ".join(f"{c}:{n}" for c, n in row["types"].most_common())
    print(f"{prio:<16} {row['files']:>6} {fmt_size(row['size']):>10} {cats}")

In [ ]:
# ── 6c. Token cost by filtering scenario ──
def token_cost(paths):
    return sum(estimate_tokens(read_text(p)) for p in paths)

all_json_tokens = token_cost(json_files)
high_only = [f for c, fs in json_by_type.items() for f in fs if RAG_PRIORITY.get(c) == "high"]
high_medium = [f for c, fs in json_by_type.items() for f in fs if RAG_PRIORITY.get(c) in {"high", "medium"}]

scenarios = {
    "all JSON":         json_files,
    "high priority":    high_only,
    "high + medium":    high_medium,
}

print(f"{'Scenario':<24} {'Files':>6} {'~Tokens':>10} {'Savings':>10}")
print("-" * 54)
for name, files in scenarios.items():
    tokens = token_cost(files)
    saving = all_json_tokens - tokens
    print(f"{name:<24} {len(files):>6} {tokens:>10} {saving:>10}")

print(f"\nDecision: excluding test data and tooling saves ~{all_json_tokens - token_cost(high_medium):,} tokens")

---
## 7. Source code signals

Quick semantic density scan of JS and Vue files to inform chunking strategy.

In [ ]:
CODE_EXTENSIONS = {".js", ".mjs", ".cjs", ".vue"}
zones_code = ["core", "map", "vite", "test", "extras"]
stats_code = []
for zone in zones_code:
    files = [f for f in by_zone.get(zone, []) if f.suffix.lower() in CODE_EXTENSIONS]
    lines = exports = functions = classes = 0
    for f in files:
        content = read_text(f)
        lines += len(content.splitlines())
        exports += len(re.findall(r"^export\s", content, re.MULTILINE))
        functions += len(re.findall(r"^(?:export\s+)?(?:async\s+)?function\s+", content, re.MULTILINE))
        classes += len(re.findall(r"^(?:export\s+)?class\s+", content, re.MULTILINE))
    stats_code.append({"zone": zone, "files": len(files), "lines": lines,
                       "exports": exports, "functions": functions, "classes": classes})

print(f"{'Zone':<10} {'Files':>8} {'Lines':>8} {'Exports':>8} {'Functions':>10} {'Classes':>8}")
print("-" * 60)
for row in stats_code:
    print(f"{row['zone']:<10} {row['files']:>8} {row['lines']:>8} {row['exports']:>8} {row['functions']:>10} {row['classes']:>8}")

In [ ]:
# Semantically densest source files
density = []
for zone in ["core", "map"]:
    for f in by_zone.get(zone, []):
        if f.suffix.lower() not in {".js", ".mjs", ".cjs"}:
            continue
        content = read_text(f)
        exports = len(re.findall(r"^export\s", content, re.MULTILINE))
        functions = len(re.findall(r"^(?:export\s+)?(?:async\s+)?function\s+", content, re.MULTILINE))
        classes = len(re.findall(r"^(?:export\s+)?class\s+", content, re.MULTILINE))
        score = exports + functions + classes
        density.append({"file": str(f.relative_to(DATA_DIR)), "score": score,
                        "exports": exports, "functions": functions, "classes": classes,
                        "lines": len(content.splitlines())})

print("Top 15 semantically densest source files:")
for row in sorted(density, key=lambda x: x["score"], reverse=True)[:15]:
    print(f"  {row['file']:<50} score={row['score']:>3} "
          f"exp={row['exports']:>2} fn={row['functions']:>2} cls={row['classes']:>2} lines={row['lines']}")

---
## 8. Summary and decisions

### Key findings

| Finding | Detail |
|---|---|
| Total retained files | ~850, ~11 MB |
| Dominant content | JS source code (`core`, `map`), then Vue SFCs, then Markdown docs |
| Documentation | 53 MD files, ~78k tokens, well-structured with H1/H2/H3 hierarchy |
| JSON | 26 files but 3.5 MB — mostly test fixtures; only schemas (8 files) are high-value |
| Content families | 8 distinct families needing different chunking strategies |

### Decisions carried forward to NB02

1. **Markdown chunking**: split at `##`, re-split large sections at `###`, merge tiny sections
2. **JSON filtering**: include schemas and i18n; exclude test fixtures and package.json
3. **JS/Vue chunking**: needs dedicated analysis — dense utility files require symbol-level splitting
4. **Barrel files** (`index.js`): metadata/navigation only, not primary chunks
5. **Test files**: exclude from initial index

In [ ]:
print("=" * 72)
print("SUMMARY — Corpus Discovery & Inventory")
print("=" * 72)
print(f"""
SCAN:
  Raw files (after dir exclusions): {len(all_files_raw)}
  Retained after file/ext exclusions: {len(all_files)}
  Excluded: {excluded_count} files ({fmt_size(excluded_size)})

ZONES: {', '.join(sorted(by_zone.keys()))}

CONTENT FAMILIES:
""")
for family, files in sorted(by_family.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"  {family:<24} {len(files):>4} files")

print(f"""
DOCUMENTATION:
  Markdown files: {len(md_files)}
  Total tokens: ~{total_doc_tokens:,}
  H2 chunks (simulation): {len(all_chunks)} (too small: {len(small)}, too large: {len(large)})

JSON:
  Total files: {len(json_files)}
  High-value (schemas): {len(high_only)} files
  Tokens saved by filtering: ~{all_json_tokens - token_cost(high_medium):,}

NEXT: NB02 formalizes the stratification and chunking hypotheses.
""")